# Stage 1 — Data

**Goal:** understand the corpus before committing to anything downstream, and
learn to work with a dataset larger than the disk you have.

TinyStories is ~7.6 GB. We never download it. `streaming=True` pulls rows over
HTTP on demand, which is how you handle any corpus that outgrows local storage —
and at real scale, *every* corpus does.

### Why this dataset makes the project possible

A 15.7M-parameter model trained on general web text produces noise. There isn't
enough capacity to learn the long tail of English, let alone facts.

TinyStories sidesteps that. It's synthetic short stories written with the
vocabulary of a 3–4 year old, so the distribution is narrow enough that a tiny
model can actually become *fluent* in it rather than merely less-bad. The
[TinyStories paper](https://arxiv.org/abs/2305.07759) is the origin of this idea.

The tradeoff is honest and worth stating up front: the model will write
grammatical, coherent children's stories and will be useless at everything else.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Set this to YOUR GitHub repo once; every notebook uses the same cell.
REPO_URL = "https://github.com/pythonstudentiam/e2e_llm_demo.git"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/content/e2e_llm_demo")
WORK = Path("/content/work")          # scratch: data + checkpoints (ephemeral!)
WORK.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

sys.path.insert(0, str(REPO / "src"))

# Colab ships torch; these are the rest. -q to keep the log readable.
%pip install -q sentencepiece "datasets>=3.0" "transformers>=4.45" "huggingface_hub>=0.30"

# HF token from the Colab Secrets panel (key icon, left sidebar). Name it
# HF_TOKEN and enable Notebook access -- the grant is PER NOTEBOOK, so every
# notebook asks separately. Never paste a token into a cell.
#
# login() rather than just setting the env var: it writes the token where every
# huggingface_hub call looks, including ones that ignore the environment.
try:
    from google.colab import userdata
    from huggingface_hub import login

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF_TOKEN loaded from Colab Secrets, authenticated")
except Exception as e:
    print("=" * 72)
    print(f"  HF_TOKEN IS NOT AVAILABLE  ({type(e).__name__}: {e})")
    print()
    print("  Every Hub call in this notebook will fail with 401 Unauthorized.")
    print("  Fix: click the key icon in the left sidebar, turn on Notebook")
    print("       access for HF_TOKEN, then RE-RUN THIS CELL before continuing.")
    print("=" * 72)

import torch
print(f"torch {torch.__version__} | CUDA {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
from tinyllm import config
from tinyllm.config import (
    model_cfg, train_cfg, data_cfg, tok_cfg, sft_cfg, gen_cfg, quant_cfg, serve_cfg, hub,
)

print(config.summary())

## 1.1 — Look at the actual data

Before any statistics, read a few rows. A surprising number of data bugs are
visible to the naked eye in the first thirty seconds and invisible in aggregates.

In [ ]:
from tinyllm.data import stream_texts

for i, text in enumerate(stream_texts(split="train", limit=3)):
    print(f"{'=' * 78}\nDOCUMENT {i}\n{'=' * 78}")
    print(text)
    print()

## 1.2 — Length distribution

The number that matters is what fraction of documents fit inside our 512-token
context. Documents longer than the window get split across training samples, so
the model never sees them whole — that bounds how much narrative structure it can
possibly learn.

In [ ]:
from tinyllm.data import corpus_stats

stats = corpus_stats(stream_texts(split="train", limit=20_000))
for k, v in stats.items():
    print(f"  {k:<14} {v:,.1f}" if isinstance(v, float) else f"  {k:<14} {v:,}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

lengths = [len(t.split()) for t in stream_texts(split="train", limit=20_000)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lengths, bins=60, color="#4C78A8", edgecolor="white", linewidth=0.5)
ax.axvline(np.mean(lengths), color="#E45756", linestyle="--", linewidth=2,
           label=f"mean {np.mean(lengths):.0f} words")

# ~1.3 tokens per word is the usual ratio for a small BPE vocabulary; stage 2
# measures the real number and we revisit this.
approx_ctx_words = data_cfg.seq_len / 1.3
ax.axvline(approx_ctx_words, color="#54A24B", linestyle="-", linewidth=2,
           label=f"~{data_cfg.seq_len}-token context (~{approx_ctx_words:.0f} words)")

ax.set_xlabel("words per story")
ax.set_ylabel("count")
ax.set_title("TinyStories document lengths")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

frac = np.mean(np.array(lengths) < approx_ctx_words)
print(f"\nRoughly {frac:.1%} of stories should fit whole inside the context window.")

## 1.3 — Vocabulary shape

TinyStories' defining property is its small vocabulary. Measuring it here tells
us what vocab size stage 2 should target: a BPE vocabulary much larger than the
corpus needs just produces rare tokens that never get trained.

In [ ]:
from collections import Counter

words = Counter()
n_docs = 0
for t in stream_texts(split="train", limit=20_000):
    words.update(w.strip('.,!?"\'').lower() for w in t.split())
    n_docs += 1

total = sum(words.values())
print(f"{n_docs:,} documents | {total:,} word tokens | {len(words):,} unique words\n")

# How many distinct words to cover most of the text?
cum = 0
covered = {}
for i, (_, c) in enumerate(words.most_common(), 1):
    cum += c
    for target in (0.90, 0.95, 0.99, 0.999):
        if target not in covered and cum / total >= target:
            covered[target] = i

for target, n in covered.items():
    print(f"  {target:.1%} of all word occurrences covered by the top {n:,} words")

print("\nMost common:", ", ".join(w for w, _ in words.most_common(20)))

Note how few distinct words carry almost all the text. That is the property that
lets an 8192-piece BPE vocabulary be generous rather than cramped here — and it
is why stage 2 does not reuse an off-the-shelf 128k-token vocabulary, which would
put 94% of the model's parameters into an embedding table for tokens this corpus
never contains.

## 1.4 — Write the tokenizer training sample

Stage 2 fits the BPE vocabulary on a sample of the corpus. Writing it to a flat
file now keeps the two stages cleanly separated, and lets the SentencePiece
trainer stream from disk instead of holding everything in memory.

In [ ]:
from pathlib import Path
from tinyllm.tokenizer import write_corpus_sample

corpus_file = WORK / "tokenizer_corpus.txt"
n = write_corpus_sample(stream_texts(split="train"), corpus_file)

size_mb = corpus_file.stat().st_size / 1e6
print(f"wrote {n:,} lines ({size_mb:.0f} MB) -> {corpus_file}")
print("\nfirst 2 lines:")
with corpus_file.open(encoding="utf-8") as f:
    for _ in range(2):
        print(" ", f.readline()[:160], "...")

## Stage 1 gate

- [x] Documents stream without downloading the full 7.6 GB
- [x] Length distribution known, and we know what fraction fits the context
- [x] Vocabulary is small enough to justify an 8192-piece BPE
- [x] `tokenizer_corpus.txt` written for stage 2

**A caution about `/content/work`:** it lives on the Colab runtime and disappears
when the runtime is recycled. Nothing here is precious — stage 2 can regenerate
it in a couple of minutes — but from stage 4 onward, anything worth keeping goes
to the Hub.

**Next:** `02_tokenizer.ipynb`